#  Qdrant initiation

In [1]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")

# Load points for collection

In [2]:
from glob import glob
from tqdm import tqdm
import json

COLLECTION_NAME = "test"
point_paths = glob(rf"..\collections\{COLLECTION_NAME}\*.json") # Points in JSON format, compatible with Qdrant upsertion
points = []
for id, path in tqdm(list(enumerate(point_paths))):
    with open(path, 'r', encoding='utf-8') as f:
        point = json.load(f)

    # Reformat to Qdrant point
    point = models.PointStruct(
        id=id+1,
        vector={
            'dense_siglip2': point['vector']['dense_siglip2'],
            'dense_bgem3': point['vector']['dense_bgem3'],
            'sparse_bgem3': models.SparseVector(
                indices=point['vector']['sparse_bgem3']['token_ids'],
                values=point['vector']['sparse_bgem3']['weights']
            ),
            # 'multi': point['vector']["multi"]
        },
        payload=point['payload']
    )
    points.append(point)

100%|██████████| 8115/8115 [01:40<00:00, 80.52it/s] 


In [ ]:
# Testing real upload speed for future process
# points = points * 43

# Collection configuration

In [4]:
dense_siglip2_params = models.VectorParams(
    size=1536,
    distance=models.Distance.COSINE
)

dense_bgem3_params = models.VectorParams(
    size=1024,
    distance=models.Distance.COSINE
)

multi_params = models.VectorParams(
    size=1024,
    distance=models.Distance.COSINE,
    multivector_config=models.MultiVectorConfig(
        comparator=models.MultiVectorComparator.MAX_SIM
    ),
    hnsw_config=models.HnswConfigDiff(m=0),
    on_disk=True
)

sparse_params = models.SparseVectorParams(
    modifier=models.Modifier.IDF
)

hnsw_params = models.HnswConfigDiff(
    m=32,
    ef_construct=200,
    full_scan_threshold=10000
)

In [5]:
if client.collection_exists(collection_name=COLLECTION_NAME):
    confirm_deletion = input('The collection already existed, do you want to replace it (y/n)? ')
    if confirm_deletion.lower() == 'y':
        client.delete_collection(collection_name=COLLECTION_NAME)
        print('Deletion complete. Run the command again')
    elif confirm_deletion.lower() == 'n':
        print('Deletion cancelled')
    else:
        print('Invalid input')
else:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            'dense_siglip2': dense_siglip2_params,
            'dense_bgem3': dense_bgem3_params,
        },
        sparse_vectors_config={'sparse_bgem3': sparse_params},
        hnsw_config=hnsw_params
    )

# Upsertion

In [6]:
client.upload_points(
    collection_name=COLLECTION_NAME,
    points=points,
    batch_size=256,
    parallel=32
)